In [ ]:
import json
import cv2 as cv
from pathlib import Path

# Paths
data_dir = Path.cwd() / "labeled_example_data"
labels_path = data_dir / "labels.json"

# Load labels
with open(labels_path, "r") as f:
    labels = json.load(f)

# Pick the first entry
entry = labels[0]
img_path = data_dir / entry["filename"]

# Load image
img = cv.imread(str(img_path))
h, w = img.shape[:2]

# Draw ground truth center if wheel present
if entry["is_present"] == 1.0:
    x_px = int(entry["center"][0] * w)
    y_px = int(entry["center"][1] * h)
    cv.circle(img, (x_px, y_px), 5, (0, 0, 255), -1)  # red dot
    print(f"{entry['filename']} → GT center: ({x_px}, {y_px})")
else:
    print(f"{entry['filename']} → No wheel present")

# Show the image in a window
cv.imshow("Check GT center", img)
cv.waitKey(0)
cv.destroyAllWindows()

rim_000.png → GT center: (37, 113)


In [1]:
import json
import cv2 as cv
from pathlib import Path

# Paths
base_dir = Path.cwd()  # current working directory
data_dir = base_dir / "labeled_example_data"
labels_path = data_dir / "labels.json"
output_dir = base_dir / "labeled_with_red_dot"
output_dir.mkdir(exist_ok=True)

# Load labels
with open(labels_path, "r") as f:
    labels = json.load(f)

# Loop through all entries
for entry in labels:
    img_path = data_dir / entry["filename"]
    img = cv.imread(str(img_path))
    if img is None:
        print(f"⚠️ Could not read {img_path}")
        continue

    h, w = img.shape[:2]

    # Draw red dot if wheel is present
    if entry["is_present"] == 1.0:
        x_px = int(entry["center"][0] * w)
        y_px = int(entry["center"][1] * h)
        cv.circle(img, (x_px, y_px), 5, (0, 0, 255), -1)  # red filled circle

    # Save annotated image
    out_path = output_dir / entry["filename"]
    cv.imwrite(str(out_path), img)

print(f"✅ Annotated images saved to: {output_dir}")

✅ Annotated images saved to: /Users/amrattia/projects/ATI Perception Take Home Challenge/Challenge 1 - Rim Center Location/labeled_with_red_dot


In [32]:
import json
import cv2 as cv
import numpy as np
from pathlib import Path

# --- Paths ---
base_dir = Path.cwd()  # run from project root
data_dir = base_dir / "labeled_example_data"
labels_path = data_dir / "labels.json"
output_dir = base_dir / "debug_check"
output_dir.mkdir(exist_ok=True)

# --- Load labels ---
with open(labels_path, "r") as f:
    labels = json.load(f)
labels_by_name = {e["filename"]: e for e in labels}

# --- Ellipse fitting helper ---
def fit_best_ellipse_center(img):
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    gray = cv.equalizeHist(gray)
    edges = cv.Canny(gray, 50, 150)
    cnts, _ = cv.findContours(edges, cv.RETR_LIST, cv.CHAIN_APPROX_NONE)
    best, score = None, -1
    H, W = gray.shape[:2]
    for c in cnts:
        if len(c) < 60:
            continue
        ell = cv.fitEllipse(c)
        (cx, cy), (MA, ma), _ = ell
        a, b = max(MA, ma), min(MA, ma)
        ar = b / (a + 1e-6)  # aspect ratio
        area = np.pi * 0.25 * a * b
        s = 0.6 * ar + 0.4 * (area / (H * W))
        if s > score:
            best, score = (cx, cy), s
    return best

# --- Check rim_030 ---
fname = "rim_030.png"
entry = labels_by_name[fname]
img_path = data_dir / fname
img = cv.imread(str(img_path))
if img is None:
    raise FileNotFoundError(img_path)

H, W = img.shape[:2]

# Draw ground truth center (red)
if entry["is_present"] == 1.0:
    gt_x = int(round(entry["center"][0] * W))
    gt_y = int(round(entry["center"][1] * H))
    cv.circle(img, (gt_x, gt_y), 6, (0, 0, 255), -1)
    cv.drawMarker(img, (gt_x, gt_y), (0, 0, 255), markerType=cv.MARKER_CROSS, markerSize=16, thickness=2)
    print(f"GT center (pixels): ({gt_x}, {gt_y})")
else:
    print("No wheel present in GT")

# Draw ellipse-fit center (green)
geo_center = fit_best_ellipse_center(img)
if geo_center:
    gx, gy = map(int, map(round, geo_center))
    
    gx, gy = gx+19, gy +432
    
    
    print(f"Ellipse-fit center: ({gx}, {gy})")
    # Draw a larger green circle with a thick outline
    cv.circle(img, (gx, gy), 5, (0, 255, 0), 2)
    cv.drawMarker(img, (gx, gy), (0, 255, 0), markerType=cv.MARKER_TILTED_CROSS, markerSize=20, thickness=2)
else:
    print("⚠️ No ellipse-fit center found")

# Save overlay
out_path = output_dir / f"{fname.replace('.png','')}_overlay.png"
cv.imwrite(str(out_path), img)
print(f"Overlay saved to {out_path}")

GT center (pixels): (69, 106)
Ellipse-fit center: (61, 95)
Overlay saved to /Users/amrattia/projects/ATI Perception Take Home Challenge/Challenge 1 - Rim Center Location/debug_check/rim_030_overlay.png
